# 思路


## Simple Baseline(> 0.60824)
直接跑一遍Simple Code（Score: 0.60500
Private score: 0.61300）不能达到。

Score: 0.65375
Private score: 0.65625

模型使用两层Transformer👇
```python
self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=2)
```
forward中使用encoder
```python
# out = self.encoder_layer(out)
out = self.encoder(out)
```

## Medium Baseline(> 0.70375)
Score: 0.76825
Private score: 0.76925

- 增加`d_model`的值，因为输出维度n_spks=600差距过大；
```python
d_model=224
```
- `nn.TransformerEncoderLayer`增加`dropout`参数，`dropout=0.2`；
```python
self.encoder_layer = nn.TransformerEncoderLayer(
			d_model=d_model, dim_feedforward=256, nhead=2, dropout=dropout
		)
```
- 堆叠3个transformer层；
```python
self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=3)
```
- 输出层改为一层并增加batchnorm机制；
```python
self.pred_layer = nn.Sequential(
      nn.BatchNorm1d(d_model),
      nn.Linear(d_model, n_spks),
    )
```



## Strong Baseline(>0.77750)
Score: 0.85025
Private score: 0.85025

使用 `torchaudio.models.Conformer` 作为编码器代替Transformer，提取序列特征。
ref:https://pytorch.org/audio/stable/generated/torchaudio.models.Conformer.html

注意:
- Conformer的输入形状为(batch_size, seq_len, d_model)，输出形状相同；
- `Conformer.forward()`需要额外输入`lengths`,是一个张量，表示每个输入序列的实际长度，以便模型在处理时可以忽略填充部分（padding）；
```python
lengths = torch.full((out.shape[0],),out.shape[1])
```
  (1)out.shape: out 是经过 prenet 处理后的张量，形状为 (batch_size, seq_len, d_model)。

  - batch_size：批次大小。

  - seq_len：序列长度（时间步数）。

  - d_model：特征维度。

  - out.shape[0] 是批次大小（batch_size）。

  - out.shape[1] 是序列长度（seq_len）。
  
  (2)torch.full: torch.full 是 PyTorch 中的一个函数，用于创建一个填充了指定值的张量。

  语法：torch.full(size, fill_value, ...)。

  size：张量的形状。

  fill_value：填充的值。
  
  (3) (out.shape[0],):
  - 这是一个元组，表示张量的形状。
  - (out.shape[0],) 表示一个一维张量，长度为 batch_size。
  
  (4) out.shape[1]:
  这是填充的值，表示序列的长度（seq_len）。



```python
import torch
import torch.nn as nn
import torchaudio.models

class Classifier(nn.Module):
    def __init__(self, d_model=224, n_spks=600, dropout=0.2):
        super().__init__()
        # 将输入维度从 40 映射到 d_model
        self.prenet = nn.Linear(40, d_model)

        # 使用 Conformer 代替 Transformer
        self.encoder = torchaudio.models.Conformer(
            input_dim=d_model,  # 输入特征维度
            num_heads=2,  # 注意力头数
            ffn_dim=256,  # Feed-forward 层隐藏维度
            num_layers=3,  # Conformer 层数
            depthwise_conv_kernel_size=31,  # 深度卷积核大小
            dropout=dropout,
        )

        # 预测层（BatchNorm + 线性分类层）
        self.pred_layer = nn.Sequential(
            nn.BatchNorm1d(d_model),
            nn.Linear(d_model, n_spks),
        )

    def forward(self, mels):
        """
        args:
            mels: (batch_size, seq_len, 40)
        return:
            out: (batch_size, n_spks)
        """
        # 线性变换调整维度
        out = self.prenet(mels)  # (batch_size, seq_len, d_model)

        # Conformer 期望输入格式 (batch_size, seq_len, d_model)，无需 permute
        lengths = torch.full((out.shape[0],),out.shape[1]).to(device)
        out, _ = self.encoder(out, lengths)

        # Mean Pooling 取全局信息
        stats = out.mean(dim=1)  # (batch_size, d_model)

        # 通过分类层
        out = self.pred_layer(stats)  # (batch_size, n_spks)

        return out

```

## Boss Baseline(>0.86500)
Score: 0.87575
Private score: 0.87250

- 根据助教提示，直接加上Self-Attention Pooling和Additive Margin Softmax,训练15万个step，结果就直接惨掉（Score: 0.83900 Private score: 0.83900），比Strong Baseline的score还不如。

- 只使用Conformer+Self-Attention Pooling，结果略有提升，但是还不能达到Boss Baseline(Score: 0.85550 Private score: 0.85550)。Additive Margin Softmax看起来实际没有起到什么作用。

- 调整Conformer头数、层数, dropout等参数，score略有波动，但不明显。

- 模型增加BatchNorm层、l2正则，改进Pooling层（Mean + Self-Attention）让池化更稳定，score略有增加。

至此，单模还是不能达到Boss Baseline，跟助教的说法不一样，不知道是代码还是参数的问题，如果能有改进的建议，非常感谢 : )

最终只好使用投票法Ensemble，才超过Boss Baseline。

**Self-Attention Pooling**

```python

class SelfAttentionPooling(nn.Module):
    """
    Mean Pooling + Self-Attention Pooling
    """
    def __init__(self, input_dim):
        super(SelfAttentionPooling, self).__init__()
        self.attention = nn.Linear(input_dim, 1)

    def forward(self, batch_rep):
        """
        input:
            batch_rep : size (N, T, H), N: batch size, T: sequence length, H: Hidden dimension

        return:
            utter_rep: size (N, H)
        """
        # Self-Attention Weights
        att_w = F.softmax(self.attention(batch_rep), dim=1)  # (N, T, 1)
        att_out = torch.sum(batch_rep * att_w, dim=1)  # Weighted sum (N, H)
        
        # Combine both
        utter_rep = att_out
        return utter_rep

```

**Additive Margin Softmax**
```python

class AMSoftmax(nn.Module):
    def __init__(self, in_features, n_classes, s=20, m=0.2): # s=30, m=0.4
        super(AMSoftmax, self).__init__()
        self.in_features = in_features
        self.n_classes = n_classes
        self.s = s  # 缩放因子
        self.m = m  # 角度边距
        self.weight = nn.Parameter(torch.FloatTensor(n_classes, in_features))
        nn.init.xavier_uniform_(self.weight)  # 初始化权重

    def forward(self, x, labels=None):
        """
        x: (batch_size, in_features) 输入特征
        labels: (batch_size,) 目标标签 (训练模式) 或 None (推理模式)
        return: (batch_size, n_classes) AM-Softmax logits
        """
        # 归一化输入和权重
        x = F.normalize(x, p=2, dim=-1)  # 归一化输入特征
        weight = F.normalize(self.weight, p=2, dim=-1)  # 归一化权重

        # 计算余弦相似度
        cosine = F.linear(x, weight)  # (batch_size, n_classes)

        if labels is not None:
            # 训练模式：计算 AM-Softmax
            one_hot = F.one_hot(labels.to(torch.int64), num_classes=self.n_classes).float()
            cosine_m = cosine - self.m * one_hot  # 仅对目标类施加 margin
            logits = self.s * cosine_m
        else:
            # 推理模式：仅进行归一化计算
            logits = self.s * cosine  # 直接返回 softmax logits

        return logits

```

# Task description
- Classify the speakers of given features.
- Main goal: Learn how to use transformer.
- Baselines:
  - Easy: Run sample code and know how to use transformer.
  - Medium: Know how to adjust parameters of transformer.
  - Strong: Construct [conformer](https://arxiv.org/abs/2005.08100) which is a variety of transformer.
  - Boss: Implement [Self-Attention Pooling](https://arxiv.org/pdf/2008.01077v1.pdf) & [Additive Margin Softmax](https://arxiv.org/pdf/1801.05599.pdf) to further boost the performance.

- Other links
  - Kaggle: [link](https://www.kaggle.com/t/ac77388c90204a4c8daebeddd40ff916)
  - Slide: [link](https://docs.google.com/presentation/d/1HLAj7UUIjZOycDe7DaVLSwJfXVd3bXPOyzSb6Zk3hYU/edit?usp=sharing)
  - Data: [link](https://drive.google.com/drive/folders/1vI1kuLB-q1VilIftiwnPOCAeOOFfBZge?usp=sharing)

# Download dataset
- Data is [here](https://drive.google.com/drive/folders/1vI1kuLB-q1VilIftiwnPOCAeOOFfBZge?usp=sharing)

## Download dataset

## Fix Random Seed

In [1]:
import numpy as np
import torch
import random

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(87)

# Data

## Dataset
- Original dataset is [Voxceleb2](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox2.html).
- The [license](https://creativecommons.org/licenses/by/4.0/) and [complete version](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/files/license.txt) of Voxceleb2.
- We randomly select 600 speakers from Voxceleb2.
- Then preprocess the raw waveforms into mel-spectrograms.

- Args:
  - data_dir: The path to the data directory.
  - metadata_path: The path to the metadata.
  - segment_len: The length of audio segment for training.
- The architecture of data directory \\
  - data directory \\
  |---- metadata.json \\
  |---- testdata.json \\
  |---- mapping.json \\
  |---- uttr-{random string}.pt \\

- The information in metadata
  - "n_mels": The dimention of mel-spectrogram.
  - "speakers": A dictionary.
    - Key: speaker ids.
    - value: "feature_path" and "mel_len"


For efficiency, we segment the mel-spectrograms into segments in the traing step.

In [3]:
import os
import json
import torch
import random
from pathlib import Path
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class myDataset(Dataset):
	def __init__(self, data_dir, segment_len=128):
		self.data_dir = data_dir
		self.segment_len = segment_len

		# Load the mapping from speaker neme to their corresponding id.
		mapping_path = Path(data_dir) / "mapping.json"
		mapping = json.load(mapping_path.open())
		self.speaker2id = mapping["speaker2id"]

		# Load metadata of training data.
		metadata_path = Path(data_dir) / "metadata.json"
		metadata = json.load(open(metadata_path))["speakers"]

		# Get the total number of speaker.
		self.speaker_num = len(metadata.keys())
		self.data = []
		for speaker in metadata.keys():
			for utterances in metadata[speaker]:
				self.data.append([utterances["feature_path"], self.speaker2id[speaker]])

	def __len__(self):
			return len(self.data)

	def __getitem__(self, index):
		feat_path, speaker = self.data[index]
		# Load preprocessed mel-spectrogram.
		mel = torch.load(os.path.join(self.data_dir, feat_path))

		# Segmemt mel-spectrogram into "segment_len" frames.
		if len(mel) > self.segment_len:
			# Randomly get the starting point of the segment.
			start = random.randint(0, len(mel) - self.segment_len)
			# Get a segment with "segment_len" frames.
			mel = torch.FloatTensor(mel[start:start+self.segment_len])
		else:
			mel = torch.FloatTensor(mel)
		# Turn the speaker id into long for computing loss later.
		speaker = torch.FloatTensor([speaker]).long()
            
		return mel, speaker

	def get_speaker_number(self):
		return self.speaker_num

## Dataloader
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.

In [4]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence


def collate_batch(batch):
	# Process features within a batch.
	"""Collate a batch of data."""
	mel, speaker = zip(*batch)
	# Because we train the model batch by batch, we need to pad the features in the same batch to make their lengths the same.
	mel = pad_sequence(mel, batch_first=True, padding_value=-20)    # pad log 10^(-20) which is very small value.
	# mel: (batch size, length, 40)
	return mel, torch.FloatTensor(speaker).long()


def get_dataloader(data_dir, batch_size, n_workers):
	"""Generate dataloader"""
	dataset = myDataset(data_dir)
	speaker_num = dataset.get_speaker_number()
	# Split dataset into training dataset and validation dataset
	trainlen = int(0.9 * len(dataset))
	lengths = [trainlen, len(dataset) - trainlen]
	trainset, validset = random_split(dataset, lengths)

	train_loader = DataLoader(
		trainset,
		batch_size=batch_size,
		shuffle=True,
		drop_last=True,
		num_workers=n_workers,
		pin_memory=True,
		collate_fn=collate_batch,
	)
	valid_loader = DataLoader(
		validset,
		batch_size=batch_size,
		num_workers=n_workers,
		drop_last=True,
		pin_memory=True,
		collate_fn=collate_batch,
	)

	return train_loader, valid_loader, speaker_num

# Model
- TransformerEncoderLayer:
  - Base transformer encoder layer in [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
  - Parameters:
    - d_model: the number of expected features of the input (required).

    - nhead: the number of heads of the multiheadattention models (required).

    - dim_feedforward: the dimension of the feedforward network model (default=2048).

    - dropout: the dropout value (default=0.1).

    - activation: the activation function of intermediate layer, relu or gelu (default=relu).

- TransformerEncoder:
  - TransformerEncoder is a stack of N transformer encoder layers
  - Parameters:
    - encoder_layer: an instance of the TransformerEncoderLayer() class (required).

    - num_layers: the number of sub-encoder-layers in the encoder (required).

    - norm: the layer normalization component (optional).

## Self-Attention Pooling (Boss)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttentionPooling(nn.Module):
    """
    Mean Pooling + Self-Attention Pooling
    """
    def __init__(self, input_dim):
        super(SelfAttentionPooling, self).__init__()
        self.attention = nn.Linear(input_dim, 1)

    def forward(self, batch_rep):
        """
        input:
            batch_rep : size (N, T, H), N: batch size, T: sequence length, H: Hidden dimension

        return:
            utter_rep: size (N, H)
        """
        # Self-Attention Weights
        att_w = F.softmax(self.attention(batch_rep), dim=1)  # (N, T, 1)
        att_out = torch.sum(batch_rep * att_w, dim=1)  # Weighted sum (N, H)
        
        # Mean Pooling
        mean_out = batch_rep.mean(dim=1)  # (N, H)
        
        # Combine both
        utter_rep = att_out + mean_out
        return utter_rep


## Additive Margin Softmax（Boss）

In [6]:
import torch
import torch.nn.functional as F

class AMSoftmax(nn.Module):
    def __init__(self, in_features, n_classes, s=20, m=0.2): # s=30, m=0.4
        super(AMSoftmax, self).__init__()
        self.in_features = in_features
        self.n_classes = n_classes
        self.s = s  # 缩放因子
        self.m = m  # 角度边距
        self.weight = nn.Parameter(torch.FloatTensor(n_classes, in_features))
        nn.init.xavier_uniform_(self.weight)  # 初始化权重

    def forward(self, x, labels=None):
        """
        x: (batch_size, in_features) 输入特征
        labels: (batch_size,) 目标标签 (训练模式) 或 None (推理模式)
        return: (batch_size, n_classes) AM-Softmax logits
        """
        # 归一化输入和权重
        x = F.normalize(x, p=2, dim=-1)  # 归一化输入特征
        weight = F.normalize(self.weight, p=2, dim=-1)  # 归一化权重

        # 计算余弦相似度
        cosine = F.linear(x, weight)  # (batch_size, n_classes)

        if labels is not None:
            # 训练模式：计算 AM-Softmax
            one_hot = F.one_hot(labels.to(torch.int64), num_classes=self.n_classes).float()
            cosine_m = cosine - self.m * one_hot  # 仅对目标类施加 margin
            logits = self.s * cosine_m
        else:
            # 推理模式：仅进行归一化计算
            logits = self.s * cosine  # 直接返回 softmax logits

        return logits


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Info]: Use {device} now!")

import torch
import torchaudio.models

class Classifier(nn.Module):
    def __init__(self, d_model=224, n_spks=600, dropout=0.2):
        super().__init__()
        # 将输入维度从 40 映射到 d_model
        self.prenet = nn.Linear(40, d_model)

        # 使用 Conformer 代替 Transformer
        self.encoder = torchaudio.models.Conformer(
            input_dim=d_model,  # 输入特征维度
            num_heads=2,  # 注意力头数
            ffn_dim=256,  # Feed-forward 层隐藏维度
            num_layers=4,  # Conformer 层数 3
            depthwise_conv_kernel_size=31,  # 深度卷积核大小
            dropout=dropout,
        )


        # Self-Attention Pooling
        self.pooling = SelfAttentionPooling(input_dim=d_model)

        self.bn = nn.BatchNorm1d(d_model)  # 归一化
        # AM-Softmax 分类层
        self.am_softmax = AMSoftmax(in_features=d_model, n_classes=n_spks)


    def forward(self, mels, labels=None):
        """
        args:
            mels: (batch_size, seq_len, 40)
            labels: (batch_size,), 可选
        return:
            如果有 labels，返回 (logits, loss, accuracy)
            如果没有 labels，返回 logits
        """
        # 线性变换调整维度
        out = self.prenet(mels)  # (batch_size, seq_len, d_model)

        # Conformer 期望输入格式 (batch_size, seq_len, d_model)，无需 permute
        lengths = torch.full((out.shape[0],), out.shape[1]).to(mels.device)
        out, _ = self.encoder(out, lengths)

        # Self-Attention Pooling 取全局信息
        stats = self.pooling(out)  # (batch_size, d_model)

        stats = self.bn(stats)  # 归一化
        
        logits = self.am_softmax(stats, labels)
        return logits

[Info]: Use cpu now!


# Learning rate schedule
- For transformer architecture, the design of learning rate schedule is different from that of CNN.
- Previous works show that the warmup of learning rate is useful for training models with transformer architectures.
- The warmup schedule
  - Set learning rate to 0 in the beginning.
  - The learning rate increases linearly from 0 to initial learning rate during warmup period.

In [9]:
import math

import torch
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR


def get_cosine_schedule_with_warmup(
	optimizer: Optimizer,
	num_warmup_steps: int,
	num_training_steps: int,
	num_cycles: float = 0.5,
	last_epoch: int = -1,
):
	"""
	Create a schedule with a learning rate that decreases following the values of the cosine function between the
	initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
	initial lr set in the optimizer.

	Args:
		optimizer (:class:`~torch.optim.Optimizer`):
		The optimizer for which to schedule the learning rate.
		num_warmup_steps (:obj:`int`):
		The number of steps for the warmup phase.
		num_training_steps (:obj:`int`):
		The total number of training steps.
		num_cycles (:obj:`float`, `optional`, defaults to 0.5):
		The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
		following a half-cosine).
		last_epoch (:obj:`int`, `optional`, defaults to -1):
		The index of the last epoch when resuming training.

	Return:
		:obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
	"""
	def lr_lambda(current_step):
		# Warmup
		if current_step < num_warmup_steps:
			return float(current_step) / float(max(1, num_warmup_steps))
		# decadence
		progress = float(current_step - num_warmup_steps) / float(
			max(1, num_training_steps - num_warmup_steps)
		)
		return max(
			0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
		)

	return LambdaLR(optimizer, lr_lambda, last_epoch)

# Model Function
- Model forward function.

In [11]:
import torch


def model_fn(batch, model, criterion, device):
	"""Forward a batch through the model."""

	mels, labels = batch
	mels = mels.to(device)
	labels = labels.to(device)

	outs = model(mels, labels)

	loss = criterion(outs, labels)

	# Get the speaker id with highest probability.
	preds = outs.argmax(1)
	# Compute accuracy.
	accuracy = torch.mean((preds == labels).float())

	return loss, accuracy

# Validate
- Calculate accuracy of the validation set.

In [12]:
from tqdm import tqdm
import torch


def valid(dataloader, model, criterion, device):
	"""Validate on validation set."""

	model.eval()
	running_loss = 0.0
	running_accuracy = 0.0
	pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit=" uttr")

	for i, batch in enumerate(dataloader):
		with torch.no_grad():
			loss, accuracy = model_fn(batch, model, criterion, device)
			running_loss += loss.item()
			running_accuracy += accuracy.item()

		pbar.update(dataloader.batch_size)
		pbar.set_postfix(
			loss=f"{running_loss / (i+1):.2f}",
			accuracy=f"{running_accuracy / (i+1):.2f}",
		)

	pbar.close()
	model.train()

	return running_accuracy / len(dataloader)

# Main function

In [15]:
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split


def parse_args():
	"""arguments"""
	config = {
		"data_dir": "/kaggle/input/ml2022spring-hw4/Dataset",
		"save_path": "model.ckpt",
		"batch_size": 32,
		"n_workers": 8,
		"valid_steps": 2000,
		"warmup_steps": 1000,
		"save_steps": 10000, # 
		"total_steps": 150000, # 70000
	}

	return config


def main(
	data_dir,
	save_path,
	batch_size,
	n_workers,
	valid_steps,
	warmup_steps,
	total_steps,
	save_steps,
):
	"""Main function."""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	print(f"[Info]: Use {device} now!")

	train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers)
	train_iterator = iter(train_loader)
	print(f"[Info]: Finish loading data!",flush = True)

	model = Classifier(n_spks=speaker_num).to(device)
	RESUME = False
	if RESUME:
		model.load_state_dict(torch.load('/kaggle/input/hw4-model/model.ckpt', map_location=device))

	criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
	optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4) # L2 正则化
	scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
	print(f"[Info]: Finish creating model!",flush = True)

	best_accuracy = -1.0
	best_state_dict = None

	pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

	for step in range(total_steps):
		# Get data
		try:
			batch = next(train_iterator)
		except StopIteration:
			train_iterator = iter(train_loader)
			batch = next(train_iterator)

		loss, accuracy = model_fn(batch, model, criterion, device)
		batch_loss = loss.item()
		batch_accuracy = accuracy.item()

		# Updata model
		loss.backward()
		optimizer.step()
		scheduler.step()
		optimizer.zero_grad()

		# Log
		pbar.update()
		pbar.set_postfix(
			loss=f"{batch_loss:.2f}",
			accuracy=f"{batch_accuracy:.2f}",
			step=step + 1,
		)

		# Do validation
		if (step + 1) % valid_steps == 0:
			pbar.close()

			valid_accuracy = valid(valid_loader, model, criterion, device)

			# keep the best model
			if valid_accuracy > best_accuracy:
				best_accuracy = valid_accuracy
				best_state_dict = model.state_dict()

			pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

		# Save the best model so far.
		if (step + 1) % save_steps == 0 and best_state_dict is not None:
			torch.save(best_state_dict, save_path)
			pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

	pbar.close()


if __name__ == "__main__":
	main(**parse_args())

[Info]: Use cpu now!


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
<ipython-input-3-55ef1f618319>:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary o

[Info]: Finish loading data!


<ipython-input-3-55ef1f618319>:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mel = torch.load(os.path.join(self.data_dir, feat_path))
<ipython-input-3-55ef1f618319>:37: 

[Info]: Finish creating model!


Train:  45% 901/2000 [17:05<21:12,  1.16s/ step, accuracy=0.00, loss=20.58, step=901]

KeyboardInterrupt: 

# Inference

## Dataset of inference

In [ ]:
import os
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset


class InferenceDataset(Dataset):
	def __init__(self, data_dir):
		testdata_path = Path(data_dir) / "testdata.json"
		metadata = json.load(testdata_path.open())
		self.data_dir = data_dir
		self.data = metadata["utterances"]

	def __len__(self):
		return len(self.data)

	def __getitem__(self, index):
		utterance = self.data[index]
		feat_path = utterance["feature_path"]
		mel = torch.load(os.path.join(self.data_dir, feat_path))

		return feat_path, mel


def inference_collate_batch(batch):
	"""Collate a batch of data."""
	feat_paths, mels = zip(*batch)

	return feat_paths, torch.stack(mels)

## Main funcrion of Inference

In [ ]:
import json
import csv
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader

def parse_args():
	"""arguments"""
	config = {
		"data_dir": "/kaggle/input/ml2022spring-hw4/Dataset",
		"model_path": "./model.ckpt",
		"output_path": "./output.csv",
	}

	return config


def main(
	data_dir,
	model_path,
	output_path,
):
	"""Main function."""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	print(f"[Info]: Use {device} now!")

	mapping_path = Path(data_dir) / "mapping.json"
	mapping = json.load(mapping_path.open())

	dataset = InferenceDataset(data_dir)
	dataloader = DataLoader(
		dataset,
		batch_size=1,
		shuffle=False,
		drop_last=False,
		num_workers=8,
		collate_fn=inference_collate_batch,
	)
	print(f"[Info]: Finish loading data!",flush = True)

	speaker_num = len(mapping["id2speaker"])
	model = Classifier(n_spks=speaker_num).to(device)
	model.load_state_dict(torch.load(model_path))
	model.eval()
	print(f"[Info]: Finish creating model!",flush = True)

	results = [["Id", "Category"]]
	for feat_paths, mels in tqdm(dataloader):
		with torch.no_grad():
			mels = mels.to(device)
			outs = model(mels)
			preds = outs.argmax(1).cpu().numpy()
			for feat_path, pred in zip(feat_paths, preds):
				results.append([feat_path, mapping["id2speaker"][str(pred)]])

	with open(output_path, 'w', newline='') as csvfile:
		writer = csv.writer(csvfile)
		writer.writerows(results)


if __name__ == "__main__":
	main(**parse_args())